# Investigating RealtimeSTT

This notebook explores the [RealtimeSTT](https://github.com/KoljaB/RealtimeSTT) library to test if it can provide lower latency transcription for the live buzzword counter.

RealtimeSTT is built on top of `faster-whisper` (by default) and includes built-in Voice Activity Detection (VAD) and realtime text updates, making it a great candidate for our live odometer.

In [ ]:
# 1. Install Dependencies
# Note: macOS users might need 'brew install portaudio' if PyAudio isn't installed.
!pip install "RealtimeSTT[faster-whisper]"

In [ ]:
import ssl
import torch

# Fix for macOS SSL certificate issues when downloading models from torch hub
ssl._create_default_https_context = ssl._create_unverified_context

# Fix for PyTorch untrusted repository errors
if hasattr(torch.hub, "_check_repo_is_trusted"):
    torch.hub._check_repo_is_trusted = lambda *args, **kwargs: True


from RealtimeSTT import AudioToTextRecorder
import time
from IPython.display import clear_output

### Basic Usage with Realtime Callbacks
Here we define a simple callback function that prints partial transcripts as you speak, allowing for "instant" feedback before the final refined sentence is returned.

In [ ]:
start_time = 0
first_update_time = None

def on_vad_start():
    global start_time, first_update_time
    start_time = time.time()
    first_update_time = None
    print("\n[VAD detected speech. Capturing...]")

def process_realtime_text(text):
    global start_time, first_update_time
    if first_update_time is None and start_time > 0:
        first_update_time = time.time()
        print(f"\r[First live update in {first_update_time - start_time:.2f}s]")
    print(f"\r[Live] {text}", end="", flush=True)

print("Initializing RealtimeSTT Recorder (loading model)...")
recorder = AudioToTextRecorder(
    model="small",               # Whisper model size
    language="en",               # Force english for testing
    enable_realtime_transcription=True,
    on_realtime_transcription_update=process_realtime_text,
    on_vad_detect_start=on_vad_start,
    realtime_model_type="tiny.en", # Use a tiny model for live updates, 'small' for final
    silero_use_onnx=False,
    spinner=False
)
print("Ready!")


In [ ]:
print("\n--- Speak into your microphone! (Stop cell to exit) ---")

try:
    while True:
        # The text() method blocks until a full sentence/turn is completed and VAD detects a pause.
        final_text = recorder.text()
        if final_text:
            total_duration = time.time() - start_time
            print(f"\n=> [Final in {total_duration:.2f}s] {final_text}\n")
except KeyboardInterrupt:
    print("\n\nStopped listening.")
finally:
    recorder.shutdown()


## Conclusion & Integration Notes

If this latency feels snappier than our custom `speech_recognition` + manual chunking approach, we can easily swap it into `app.py`!

**Why it might be faster:**
RealtimeSTT uses a dual-model approach. It runs an extremely fast model (like `tiny`) constantly to fire off the `on_realtime_transcription_update` callbacks, giving you instant visual feedback. Then, when it detects you've paused speaking, it runs the higher quality model (like `small` or `medium`) over the entire chunk to give you the finalized transcript.

## Comparison: The Original Approach

Below is the exact `speech_recognition` + `faster-whisper` logic currently used in our app for a side-by-side latency comparison.

In [ ]:
import speech_recognition as sr
from faster_whisper import WhisperModel
import wave
import io
import numpy as np
import time

print("Loading Whisper model (small)...")
faster_model = WhisperModel("small", device="auto", compute_type="default")
recognizer = sr.Recognizer()

# Ensure safety buffer doesn't erode
recognizer.dynamic_energy_threshold = False  
recognizer.energy_threshold = 400

In [ ]:
print("\n--- Speak into your microphone! (Stop cell to exit) ---")
print("Calibrating...")
with sr.Microphone() as source:
    recognizer.adjust_for_ambient_noise(source, duration=2)
    print("Ready!")
    
    try:
        while True:
            try:
                # This listens until you pause (based on energy thresholds)
                start_time = time.time()
                audio = recognizer.listen(source, timeout=1)
                listen_duration = time.time() - start_time
                
                print(f"[Audio captured in {listen_duration:.2f}s, Transcribing...]")
                
                # Convert to raw float32 for faster-whisper
                wav_bytes = audio.get_wav_data(convert_rate=16000, convert_width=2)
                with wave.open(io.BytesIO(wav_bytes), "rb") as wf:
                    frames = wf.readframes(wf.getnframes())
                    audio_array = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
                
                # Transcribe
                transcribe_start = time.time()
                segments, _ = faster_model.transcribe(audio_array, condition_on_previous_text=False)
                text = " ".join([segment.text for segment in segments]).strip()
                transcribe_duration = time.time() - transcribe_start
                
                if text:
                    print(f"=> [Final in {transcribe_duration:.2f}s] {text}\n")
                
            except sr.WaitTimeoutError:
                continue
            except (sr.UnknownValueError, sr.RequestError):
                continue
    except KeyboardInterrupt:
        print("\n\nStopped listening.")